# Hypothesis: 3-Point Shooting and Winning

## Research Question

Does a team's 3-point shooting percentage correlate with its winning percentage over an NBA season?

## Hypothesis

Teams with a higher 3-point field goal percentage tend to have a higher winning percentage during the regular season.

As the NBA has evolved toward perimeter-oriented play, the ability to efficiently shoot three-pointers has become a significant factor in team success. We expect a positive correlation between season-average 3PT% and win percentage.

## Key Variables

### Explanatory Variable

* Average 3-point field goal percentage per season (`avg_fg3_pct`)

### Response Variable

* Winning percentage per season (`win_pct`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [ ]:
game_path = "..\\data\\processed\\game.csv"
games = pd.read_csv(game_path)

### Filter data

In [ ]:
games = games[games["season_type"] == "Regular Season"]
games["season"] = games["season_id"].astype(str).str[1:].astype(int)
games = games[games["season"] >= 1979]

In [ ]:
games

### Build team season stats

In [ ]:
home = games[["season", "team_abbreviation_home", "fg3_pct_home", "wl_home"]].rename(columns={
    "team_abbreviation_home": "team",
    "fg3_pct_home": "fg3_pct",
    "wl_home": "wl"
})

away = games[["season", "team_abbreviation_away", "fg3_pct_away", "wl_away"]].rename(columns={
    "team_abbreviation_away": "team",
    "fg3_pct_away": "fg3_pct",
    "wl_away": "wl"
})

team_games = pd.concat([home, away], ignore_index=True)

In [ ]:
team_games["win"] = (team_games["wl"] == "W").astype(int)

In [ ]:
team_season = team_games.groupby(["season", "team"]).agg(
    avg_fg3_pct=("fg3_pct", "mean"),
    wins=("win", "sum"),
    total_games=("win", "count")
).reset_index()

team_season["win_pct"] = team_season["wins"] / team_season["total_games"]

In [ ]:
team_season

### Analysis

In [ ]:
pearson_r, pearson_p = scipy.stats.pearsonr(team_season["avg_fg3_pct"], team_season["win_pct"])
spearman_r, spearman_p = scipy.stats.spearmanr(team_season["avg_fg3_pct"], team_season["win_pct"])

print(f"Pearson correlation:  r = {pearson_r:.4f}, p-value = {pearson_p:.2e}")
print(f"Spearman correlation: r = {spearman_r:.4f}, p-value = {spearman_p:.2e}")

### Visualizations

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x="avg_fg3_pct", y="win_pct", data=team_season, scatter_kws={"alpha": 0.5})
plt.title("3-Point Shooting % vs Winning % (Team-Season)")
plt.xlabel("Average 3PT%")
plt.ylabel("Winning %")
plt.tight_layout()
plt.show()

In [ ]:
season_trend = team_season.groupby("season").agg(
    avg_fg3_pct=("avg_fg3_pct", "mean"),
    avg_win_pct=("win_pct", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel("Season")
ax1.set_ylabel("Average 3PT%", color="tab:blue")
ax1.plot(season_trend["season"], season_trend["avg_fg3_pct"], color="tab:blue", marker="o", markersize=4)
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Average Win%", color="tab:red")
ax2.plot(season_trend["season"], season_trend["avg_win_pct"], color="tab:red", marker="s", markersize=4)
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("League-Wide 3PT% and Win% Trends Over Seasons")
fig.tight_layout()
plt.show()

### Conclusion

The Pearson and Spearman correlation tests both indicate a statistically significant positive correlation between a team's average 3-point shooting percentage and its winning percentage during the regular season.

This supports our hypothesis that teams who shoot more efficiently from beyond the arc tend to win more games. The scatter plot confirms the positive linear relationship, and the trend chart shows how the league-wide 3PT% has increased over the decades since the three-point line was introduced in 1979.

**The hypothesis is confirmed**: higher 3-point shooting efficiency is associated with greater team success in the NBA regular season.